In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
 

C:\Users\Apoorv Bagga\AppData\Roaming\Python\Python310\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
corpus = """
machine learning models learn patterns from data.
sequence models process data step by step.
recurrent neural networks are designed for sequential tasks.
rnn models maintain hidden states across time steps.
long short term memory networks solve long dependency problems.
lstm uses gates to control information flow.
gru models simplify the lstm architecture.
sequence prediction is useful in many applications.
language modeling predicts the next word in a sentence.
speech recognition processes audio sequences.
time series forecasting predicts future values.
music generation creates new melodies.
generative models learn probability distributions.
they generate new samples similar to training data.
sequence generation is widely used in artificial intelligence.
deep learning improves sequence modeling performance.
""".strip()
 

In [3]:
words = corpus.lower().split()
 
# Build vocabulary
vocab = sorted(set(words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)
 
print(f"Vocabulary size : {vocab_size}")
print(f"Total words     : {len(words)}")
print(f"Sample vocab    : {vocab[:10]}\n")
 

Vocabulary size : 88
Total words     : 111
Sample vocab    : ['a', 'across', 'applications.', 'architecture.', 'are', 'artificial', 'audio', 'by', 'control', 'creates']



In [4]:
SEQ_LEN = 5   # number of context words
 
def build_sequences(words, seq_len):
    X, y = [], []
    indices = [word2idx[w] for w in words]
    for i in range(len(indices) - seq_len):
        X.append(indices[i : i + seq_len])
        y.append(indices[i + seq_len])
    return np.array(X), np.array(y)
 
X, y = build_sequences(words, SEQ_LEN)
X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.long)
 
print(f"Training samples: {len(X)}")
print(f"Example X[0]    : {X[0]}  →  y[0]: {y[0]}\n")

Training samples: 106
Example X[0]    : [37 34 43 33 49]  →  y[0]: 19



In [5]:
# ─────────────────────────────────────────────
# 4. LSTM MODEL DEFINITION
# ─────────────────────────────────────────────
class LSTMTextGenerator(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super(LSTMTextGenerator, self).__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim)
        self.lstm       = nn.LSTM(embed_dim, hidden_dim,
                                  num_layers=num_layers,
                                  batch_first=True,
                                  dropout=0.3 if num_layers > 1 else 0)
        self.fc         = nn.Linear(hidden_dim, vocab_size)
 
    def forward(self, x, hidden=None):
        embeds = self.embedding(x)            # (batch, seq_len, embed_dim)
        out, hidden = self.lstm(embeds, hidden)
        logits = self.fc(out[:, -1, :])       # take last time-step
        return logits, hidden
 
EMBED_DIM  = 64
HIDDEN_DIM = 128
NUM_LAYERS = 2
 
model     = LSTMTextGenerator(vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)
 
print(model)
print()

LSTMTextGenerator(
  (embedding): Embedding(88, 64)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=128, out_features=88, bias=True)
)



In [7]:
# ─────────────────────────────────────────────
# 5. TRAINING
# ─────────────────────────────────────────────
EPOCHS     = 150
BATCH_SIZE = 32
 
dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
loader  = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
 
model.train()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits, _ = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # gradient clipping
        optimizer.step()
        total_loss += loss.item()
 
    if epoch % 25 == 0:
        avg_loss = total_loss / len(loader)
        print(f"Epoch [{epoch:3d}/{EPOCHS}]  Loss: {avg_loss:.4f}")
 
print("\nTraining complete!\n")

Epoch [ 25/150]  Loss: 0.0053
Epoch [ 50/150]  Loss: 0.0030
Epoch [ 75/150]  Loss: 0.0019
Epoch [100/150]  Loss: 0.0015
Epoch [125/150]  Loss: 0.0011
Epoch [150/150]  Loss: 0.0008

Training complete!



In [8]:
# ─────────────────────────────────────────────
# 6. SEQUENCE GENERATION
# ─────────────────────────────────────────────
def generate_sequence(model, seed_words, num_words=10, temperature=1.0):
    """
    Generate new words given a seed sequence.
    temperature < 1  → more deterministic
    temperature > 1  → more random / creative
    """
    model.eval()
    seed_idx = [word2idx.get(w, 0) for w in seed_words[-SEQ_LEN:]]
    generated = list(seed_words)
 
    with torch.no_grad():
        for _ in range(num_words):
            inp    = torch.tensor([seed_idx[-SEQ_LEN:]], dtype=torch.long)
            logits, _ = model(inp)
            probs  = torch.softmax(logits / temperature, dim=-1).squeeze()
            next_idx = torch.multinomial(probs, 1).item()
            generated.append(idx2word[next_idx])
            seed_idx.append(next_idx)
 
    return " ".join(generated)
 
 
print("=" * 60)
print("GENERATED SEQUENCES (LSTM)")
print("=" * 60)
 
seeds = [
    ["machine", "learning", "models", "learn", "patterns"],
    ["long",    "short",    "term",   "memory", "networks"],
    ["sequence", "models",  "process", "data",  "step"],
    ["generative", "models", "learn", "probability", "distributions"],
]
 
for seed in seeds:
    output = generate_sequence(model, seed, num_words=10, temperature=0.8)
    print(f"\nSeed   : {' '.join(seed)}")
    print(f"Output : {output}")
 
print("\n" + "=" * 60)

GENERATED SEQUENCES (LSTM)

Seed   : machine learning models learn patterns
Output : machine learning models learn patterns from data. sequence models process data step by step. recurrent

Seed   : long short term memory networks
Output : long short term memory networks solve long dependency problems. lstm uses gates to control information

Seed   : sequence models process data step
Output : sequence models process data step by step. recurrent neural networks are designed for sequential tasks.

Seed   : generative models learn probability distributions
Output : generative models learn probability distributions they generate new samples to to training data. sequence is

